# Template application to ceate confluence pages with icons

In [ ]:
overview_template = 'overview.confluence.jinja2'
list_template = 'list.templ.confluence'
source_file = 'output/fyayc_icon_library-ssot-cloudinary.yaml' 

In [ ]:
import markupsafe
header=markupsafe.Markup('<span style="color:red;"><b>For testing purposes only!</b></span> ')

In [ ]:
import re
import logging
log = logging.getLogger(__name__)

In [ ]:
import xml.dom.minidom
from lxml import etree
import IPython

# valid xml namespaces and schema for Confluence 6 storage format
# See 
xml_namespaces=[
    'xmlns="http://www.w3.org/1999/xhtml"',
    'xmlns:ac="http://www.atlassian.com/schema/confluence/4/ac/"',
    'xmlns:ri="http://www.atlassian.com/schema/confluence/4/ri/"',
    'xmlns:acxhtml="http://www.atlassian.com/schema/confluence/4/"'
]

def encapsulate_storage_format(xml):
    return '<?xml version="1.0"?><root doc="container to properly encapsulate xml" {} >\n{}\n</root>'.format(
            ' '.join(xml_namespaces), xml)

def beautify_xml(flat_xml):
    try:
        encapsulated = encapsulate_storage_format(flat_xml)
        dom = xml.dom.minidom.parseString(encapsulated)
        return dom.toprettyxml()
    except xml.parsers.expat.ExpatError as e:
        log.error('Expat error {}'.format(e))
        raise Exception(e)
    
def validate_storage_format(xml_in_storage_format):
        '''Validate if input xml conforms to Confluence storage format specifications'''
        parser = etree.XMLParser(dtd_validation=False)
        try:
            etree.fromstring(encapsulate_storage_format(xml_in_storage_format), parser)
            return None
        except xml.parsers.expat.ExpatError as e:
            m = re.search('line ([0-9]+), column ([0-9]+)', str(e))
            message = 'Malformed xml ' + flat_xml[:50] + ' ...'
            if m:
                lines = wrapped.splitlines()
                messaage = 'Malformed xml input on position {} in: {}'.format(m.group(2), lines[int(m.group(1))-1])
            else:
                message = message + flat_xml[:50] + ' ...'
            return message

beautify_xml('<a></a>')
try:
    beautify_xml('<b></a>')
    assert false, 'Should fail'
except Exception as e:
    print('all fine!')

# Load datasource

In [ ]:
import yaml

content = None
with open(source_file, 'r') as source:
     content = yaml.safe_load(source)

# Print some information on what was loaded
print('Loaded {} icons from {}'.format(len(content['icons']), source_file))

# Transform icon map into table form used by template

`date = { 'groups': { 
    'Tangible Objects' : { 
        'icons': [  i1, i2, i3, ....
     }, 
    'Persons' : {
        'icons': [ ...
    }
}

In [ ]:
import html

section_set = set( value for value in map(lambda key: content['icons'][key]['folder'], content['icons']) )
sections = { html.escape(k) :{ 'icons': [] } for k in section_set}
sections

In [ ]:
# Map all icons into the destination data structure
for key in content['icons']:
    icon = content['icons'][key]
    sections[icon['folder']]['icons'].append(icon)
    png = icon.get('png')
    if png:
        icon['url'] = png['url']
        icon['id'] = png['id']
    else:
        icon['style'] = 'style="border:2px solid orange"'
        svg = icon.get('svg', { 'url' : 'no-svg-fallback-available', 'id': '9999'})
        icon['url'] = svg['url']
        icon['fallback'] = 'svg'
        icon['id'] = svg['id']
    if not icon.get('name'):
        description = icon.get('description')
        if description:
            icon['name'] = description.get('en', '')
        else:
            icon['name'] = 'No name for icon {}'.format(icon)
    if not icon.get('id'):
        print('Icon {} has no id'.format(icon))
        icon['id'] = '9999'
    
    keywords = icon.get('keywords')
    if keywords:
        icon['search-terms'] = ', '.join([ keywords['de'], keywords['en'] ])
    

In [ ]:
for key in sections:
    icons = sections[key]['icons']
    sections[key]['icons'] = sorted(icons, key=lambda i : int(i['id']))

In [ ]:
sections['Transportation']['icons']

In [ ]:
data = { 'sections': sections }

# Sandbox for overview template

In [ ]:
entity_template = None
with open('./templates/' + overview_template, 'r') as f:
    entity_template = f.read()

assert entity_template

IPython.display.Code(entity_template)

In [ ]:
from jinja2 import Environment, FileSystemLoader, select_autoescape

env = Environment(
    loader=FileSystemLoader('./templates'),
    autoescape=select_autoescape(['html', 'xml'])
)
env.globals.update({ 'util': None, 'data': data, 'header': header })

entity_template = env.get_template(overview_template)
rendered_entity_template = entity_template.render()
confluence_page_content = re.sub('<!--.+?->(\n+)*', '', rendered_entity_template) # strip comment lines
IPython.display.Code(confluence_page_content)

In [ ]:
IPython.display.HTML('<h1>{}</h1><br/>{}'.format('Icon Library', confluence_page_content))

In [ ]:
with open('output/overview.confluence', 'w') as out:
    out.write(confluence_page_content)

In [ ]:
validate_storage_format(confluence_page_content)